# Armenian Speech-to-Text with Whisper

Transcribe recordings of **any length** with [`BuzzASR/armenian`](https://huggingface.co/BuzzASR/armenian), a Whisper model fine-tuned for Armenian.

Whisper only sees 30 s per forward pass, so long audio is split at natural pauses using Silero VAD, decoded in batches, and returned with timestamps.

![Pipeline](assets/pipeline.svg)

**What's inside:** load model → chunk with VAD → transcribe → save JSON/TXT → (optional) WER / CER.

## 1 · Install

In [51]:
%pip install -q silero-vad soundfile torch torchaudio transformers jiwer

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2 · Configuration

Edit the values below, then run the cells top to bottom.

In [52]:
import json
import re
import time

import soundfile as sf
import torch
import torchaudio

AUDIO_PATH = "audio/test_am.wav"     
OUT_PREFIX = None              
REFERENCE_TEXT = "Ոսկրային վերականգնումը խթանող կենսաբանական և կենսաֆիզիկական մեթոդները։ Ինչպես արդեն գիտենք, կոտրվածքների սերտաճումը բարդ, կասլադային և համակարգված կենսաբանական գործընթաց է, որը ներառում է տարբեր մեխանիզմներ և կարգավորվում է ինչպես տեղային, այնպես էլ համակարգային գործոններով։ Այս բարդ կասկադային գործընթացի ցանկացած փուլի խանգառում կարող է բացասաբար ազդել սերտաճման վրա և դեպքերի հինգից տաս տոկոսում հանգեցնել սերտաճման խանգարման։ Հաճախ, սերտաճման գործընթացի խանգարման պատճառը մնում է անհայտ։ Այն կարող է պայմանավորված լինել ոսկրաբեկորների անբավարար համադրմամբ և անկայունությամբ, հիվանդի ընդհանուր առողջական վիճակով, վնասվածքի բնույթով և ծանրությամբ, ինչպես նաև տեղային արյունամատակարարման խանգառումներով։"         

MODEL_NAME = "BuzzASR/armenian"
SR = 16000
MAX_CHUNK_S = 28.0             
BATCH_SIZE = 8                 
NUM_BEAMS = 5                  
VAD_THRESHOLD = 0.35           


MEDICAL_PROMPT = (
    "Բժշկական տերմինաբանություն՝ "
    "կոճ, սրունքաթաթային հոդ, կոտրվածք, "
    "ոսկրային բեկորներ, տեղաշարժ, ռեպոզիցիա, "
    "կապանային ապարատ, փափուկ հյուսվածքներ, "
    "այտուց, արյունազեղում, գիպսային վիրակապ, "
    "օրթեզ, անշարժացում։"
)

## 3 · Load the model

In [53]:
from silero_vad import load_silero_vad, get_speech_timestamps
from transformers import WhisperForConditionalGeneration, WhisperProcessor

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME, torch_dtype=dtype)
model = model.to(device).eval()
proc = WhisperProcessor.from_pretrained(MODEL_NAME)
vad_model = load_silero_vad()

print(f"Loaded {MODEL_NAME} on {device} ({dtype})")

Loading weights: 100%|██████████| 1259/1259 [00:01<00:00, 648.96it/s]


Loaded BuzzASR/armenian on cuda (torch.float16)


## 4 · Audio loading and VAD chunking

Silero VAD finds the speech segments, then neighbouring segments are merged into chunks of at most `MAX_CHUNK_S` seconds. Cuts land in pauses instead of mid-word.

![Chunking](assets/chunking.svg)

In [54]:
def load_audio(path):
    """Load any audio file as mono, 16 kHz float32, and report its level."""
    wav, sr = sf.read(path, dtype="float32")
    if wav.ndim > 1:
        wav = wav.mean(axis=1)
    wav = torch.from_numpy(wav)
    if sr != SR:
        wav = torchaudio.functional.resample(wav, sr, SR)

    peak = wav.abs().max().item() if len(wav) else 0.0
    rms = wav.pow(2).mean().sqrt().item() if len(wav) else 0.0
    print(f"Loaded {path}: {len(wav) / SR:.1f} s | peak {peak:.4f} | rms {rms:.4f}")

    if peak < 0.005:
        print("WARNING: the audio is almost silent. Check the microphone / input device "
              "and the file itself; there is nothing for the model to transcribe.")
    elif peak < 0.5:
        gain = 0.9 / peak                 
        wav = wav * gain
        print(f"Quiet recording, boosted x{gain:.1f}")
    return wav


def make_chunks(wav, vad_model):
    """Return a list of (start_sample, end_sample), each at most MAX_CHUNK_S long."""
    speech = get_speech_timestamps(
        wav,
        vad_model,
        sampling_rate=SR,
        threshold=VAD_THRESHOLD,
        min_silence_duration_ms=400,
        speech_pad_ms=200,
        max_speech_duration_s=MAX_CHUNK_S,
    )
    if not speech:
        return []

    max_len = int(MAX_CHUNK_S * SR)
    chunks = []
    cur_start, cur_end = speech[0]["start"], speech[0]["end"]
    for seg in speech[1:]:
        if seg["end"] - cur_start <= max_len:
            cur_end = seg["end"]          
        else:
            chunks.append((cur_start, cur_end))
            cur_start, cur_end = seg["start"], seg["end"]
    chunks.append((cur_start, cur_end))
    return chunks

## 5 · Batched transcription


In [55]:
def transcribe_long(path, batch_size=BATCH_SIZE, num_beams=NUM_BEAMS):
    wav = load_audio(path)
    chunks = make_chunks(wav, vad_model)
    print(f"Audio: {len(wav) / SR:.1f} s, {len(chunks)} chunks")
    if not chunks:
        print("No speech detected, nothing to transcribe. See the level report above; "
              "if the peak is near 0 the recording is silent, otherwise try a lower VAD_THRESHOLD.")
        return []

    results = []
    for i in range(0, len(chunks), batch_size):
        batch = chunks[i:i + batch_size]
        audio_list = [wav[s:e].numpy() for s, e in batch]
        inputs = proc(
            audio_list,
            sampling_rate=SR,
            return_tensors="pt",
            return_attention_mask=True,
        )

        feats = inputs.input_features.to(device, dtype)
        attention_mask = inputs.attention_mask.to(device)

        with torch.no_grad():
            ids = model.generate(
                feats,
                attention_mask=attention_mask,
                max_new_tokens=300,
                num_beams=num_beams,
                do_sample=False,
            )
        texts = proc.batch_decode(
            ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )
        for (s, e), text in zip(batch, texts):
            results.append({
                "start": round(s / SR, 2),
                "end": round(e / SR, 2),
                "text": text.strip(),
            })
    return results

## 6 · Run

Each segment keeps its start and end time in seconds, which is what you need later to align speaker diarization.

In [56]:
t = time.time()
results = transcribe_long(AUDIO_PATH)
print(f"Done in {time.time() - t:.1f} s\n")

for r in results:
    print(f"[{r['start']:7.2f} → {r['end']:7.2f}]  {r['text']}")

full_text = " ".join(r["text"] for r in results)

if OUT_PREFIX and results:
    with open(OUT_PREFIX + ".json", "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    with open(OUT_PREFIX + ".txt", "w", encoding="utf-8") as f:
        f.write(full_text)
    print(f"\nSaved {OUT_PREFIX}.json and {OUT_PREFIX}.txt")

Loaded audio/test_am.wav: 47.0 s | peak 0.6333 | rms 0.0704


[transformers] Both `max_new_tokens` (=300) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Audio: 47.0 s, 3 chunks
Done in 2.9 s

[   0.00 →   17.86]  կենսաբանական և կենսաֆիզիկական մեթոդները ինչպես արդեն գիտենք կոտրվածքների սերտացումը բարդ կասկադային և համակարգված կենսաբանական գործընթաց է որը ներառում է տարբեր մեխանիզմներ և կարգավորում է ինչպես տեղային այնպես էլ համակարգչային գործոններով
[  17.94 →   28.52]  Այս բարդ կասկադային գործընթացի ցանկացած փուլի խանգարում կարող է բացասաբար ազդել սրտաճման վրա և դեպքերի հինգից տաս տոկոսն հանգեցնել սրտաճման խանգարման
[  28.63 →   47.00]  Հաճախ սերտանման գործընթացի խանգարման պատճառը մնում է անհայտ այն կարող է պայմանավորված լինել ոսկրաբեկորների անբավարար համադրման և անկայունությամբ հիվանդի ընդհանուր առողջական վիճակով վնասվածքի բնույթով և ծանրությամբ ինչպես նաև տեղային արյունամատակարման խանգարումներով


## 7 · Evaluate *(optional)*

Set `REFERENCE_TEXT` in the configuration cell to the ground-truth transcript to get word and character error rates. Text is lowercased and stripped of punctuation first.

In [57]:
import jiwer


def normalize(s):
    return " ".join(re.sub(r"[^\w\s]", "", s.lower()).split())


if REFERENCE_TEXT:
    ref, hyp = normalize(REFERENCE_TEXT), normalize(full_text)
    print(f"WER: {jiwer.wer(ref, hyp):.3f}")
    print(f"CER: {jiwer.cer(ref, hyp):.3f}")
else:
    print("Set REFERENCE_TEXT in the configuration cell to compute WER / CER.")

WER: 0.176
CER: 0.066
